<a href="https://colab.research.google.com/github/Monia1801/Prompt-Engineering/blob/main/Exp_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langgraph langchain-google-genai langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 30.3 MB/s eta 0:00:00
  Attempting uninstall: langchain_core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [2]:
import os
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from google.colab import userdata

In [4]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.2)
checkpointer = InMemorySaver()

In [5]:
def llm_node(state: MessagesState):
  """Generates or revises content based on the message history."""
  response = llm.invoke(state["messages"])
  return {"messages": [response]}
def human_node(state: MessagesState):
  """Displays the AI output and triggers an interrupt for feedback."""
  last_ai_message = state["messages"][-1].content
  print(f"\n--- AI Generated Content ---\n{last_ai_message[0]['text']}"           "\n----------------------------")
  feedback = interrupt(
     "Action (Type 'APPROVE' to finish, or type your revision feedback): "
  )
  return {"messages": [HumanMessage(content=feedback)]}

In [6]:
def router_logic(state: MessagesState):
  """Inspects human feedback to decide whether to exit or loop back."""
  last_user_message = state["messages"][-1].content
  if last_user_message.strip().upper() == "APPROVE":
    return "accept"
  return "revise"

In [8]:
builder = StateGraph(MessagesState)
builder.add_node("llm_worker", llm_node)
builder.add_node("human_gate", human_node)
builder.add_edge(START, "llm_worker")
builder.add_edge("llm_worker", "human_gate")
builder.add_conditional_edges(
    "human_gate",
    router_logic,
    {
        "accept": END,
        "revise": "llm_worker"
    }
)
compiled_graph = builder.compile(checkpointer=checkpointer)

In [9]:
print("--- Starting Human-in-the-Loop Loopback Graph ---")
print("Instructions: Enter a Thread ID to start or resume a session. Type 'exit' to quit.")
while True:
  thread_id = input("\nEnter Thread ID (e.g., user_1, user_2) or 'exit': ").strip()
  if thread_id.lower() == 'exit':
     print("Shutting down conversational engine.")
     break
  if not thread_id:
    continue
  config = {"configurable": {"thread_id": thread_id}}
  current_state = compiled_graph.get_state(config)
  if current_state.next and current_state.tasks and getattr(current_state.tasks[0], "interrupts", None):
       print(f" >> Resuming active thread: '{thread_id}'")
       prompt_string = current_state.tasks[0].interrupts[0].value
       user_response = input(prompt_string).strip()
       compiled_graph.invoke(Command(resume=user_response), config=config)
  else:
    print(f" >> Initializing new session for thread: '{thread_id}'")
    user_idea = input("Enter your idea: ").strip()
    if not user_idea:
      continue
    initial_input = {"messages": [HumanMessage(content=user_idea)]}
    compiled_graph.invoke(initial_input, config=config)

    post_run_state = compiled_graph.get_state(config)
    if not post_run_state.next:
      print(f" >> Workflow for thread '{thread_id}' completed successfully via APPROVE!")



--- Starting Human-in-the-Loop Loopback Graph ---
Instructions: Enter a Thread ID to start or resume a session. Type 'exit' to quit.

Enter Thread ID (e.g., user_1, user_2) or 'exit': Enter a Thread ID to start or resume a session. Type 'exit' to quit. 
 >> Initializing new session for thread: 'Enter a Thread ID to start or resume a session. Type 'exit' to quit.'
Enter your idea: Can you provide a single line quote to write in the greeting card which will be given to my mother on mother's day? 

--- AI Generated Content ---
Here are a few options, depending on the tone you want:

*   **Heartfelt:** "Everything I am, I owe to you, and I am so grateful for every bit of it."
*   **Sweet:** "To the world, you are a mother, but to me, you are the world."
*   **Appreciative:** "Thank you for being my constant support, my biggest cheerleader, and my best friend."
*   **Simple:** "Wishing a beautiful day to the woman who makes our home a place of love."
*   **Short & Classic:** "Happy Mother’s